# Lab 1 - Open quantum dynamics, tangent/transverse motion, and order sensitivity

This notebook provides a compact engineering implementation of the report's core ideas:

1. represent a qubit by a density matrix $\rho$;
2. compare a coherent rotation and amplitude damping;
3. show that the operation order can matter;
4. decompose a local velocity into orbit-tangent and spectral-block components;
5. compute a discrete Uhlmann transport factor along a closed state-space loop.

> The numerical demonstrations validate the code and intuition. They do not prove the reviewed manuscript's strongest theorems.

In [ ]:
from pathlib import Path
import sys

repo_root = Path.cwd().resolve()
if repo_root.name == "notebooks":
    repo_root = repo_root.parent
sys.path.insert(0, str(repo_root / "src"))

import numpy as np
import matplotlib.pyplot as plt

from mmals_path_memory import (
    amplitude_damping_channel,
    apply_unitary,
    bloch_to_density,
    density_to_bloch,
    discrete_uhlmann_transport,
    lyapunov_generator,
    tangent_normal_decomposition,
    trace_distance,
    von_neumann_entropy,
)

## 1. Define a state and two operations

We use an $x$-axis unitary rotation and an amplitude-damping channel. The latter contracts transverse Bloch coordinates and moves the state toward the ground state.

In [ ]:
rho0 = bloch_to_density([0.55, 0.25, -0.10])
theta = np.deg2rad(55)
probability = 0.35

ux = np.array([
    [np.cos(theta / 2), -1j * np.sin(theta / 2)],
    [-1j * np.sin(theta / 2), np.cos(theta / 2)],
], dtype=complex)

rho_rotation_then_damping = amplitude_damping_channel(apply_unitary(rho0, ux), probability)
rho_damping_then_rotation = apply_unitary(amplitude_damping_channel(rho0, probability), ux)

print("Initial Bloch vector:", density_to_bloch(rho0))
print("Rotation -> damping:", density_to_bloch(rho_rotation_then_damping))
print("Damping -> rotation:", density_to_bloch(rho_damping_then_rotation))
print("Order gap (trace distance):", trace_distance(rho_rotation_then_damping, rho_damping_then_rotation))

## 2. Visualize both trajectories in the Bloch-ball x-z projection

In [ ]:
def rotate_x(vector, angle):
    x, y, z = vector
    c, s = np.cos(angle), np.sin(angle)
    return np.array([x, c*y - s*z, s*y + c*z])


def damp_bloch(vector, p):
    x, y, z = vector
    q = np.sqrt(1-p)
    return np.array([q*x, q*y, (1-p)*z + p])

v0 = density_to_bloch(rho0)
steps = 60
rotation_path = [rotate_x(v0, theta*t) for t in np.linspace(0, 1, steps)]
rd_path = rotation_path + [damp_bloch(rotation_path[-1], probability*t) for t in np.linspace(0, 1, steps)[1:]]

damping_path = [damp_bloch(v0, probability*t) for t in np.linspace(0, 1, steps)]
dr_path = damping_path + [rotate_x(damping_path[-1], theta*t) for t in np.linspace(0, 1, steps)[1:]]

angle = np.linspace(0, 2*np.pi, 400)
plt.figure(figsize=(8, 7))
plt.plot(np.cos(angle), np.sin(angle), linewidth=1, label="Bloch boundary")
plt.plot([v[0] for v in rd_path], [v[2] for v in rd_path], linewidth=2, label="R -> D")
plt.plot([v[0] for v in dr_path], [v[2] for v in dr_path], linewidth=2, label="D -> R")
plt.scatter([v0[0]], [v0[2]], s=70, label="initial")
plt.xlabel("x")
plt.ylabel("z")
plt.axis("equal")
plt.grid(True)
plt.legend()
plt.title("Same operations, different order")
plt.show()

## 3. Tangent and spectral-block decomposition

For a non-degenerate density operator, the cross-eigenblock component is tangent to the unitary orbit. The block-diagonal component is the local spectral-change direction.

For degeneracies, the correct condition uses whole spectral projectors, not only individual diagonal entries.

In [ ]:
dt = 1e-4
rho_rotated_small = apply_unitary(rho0, np.array([
    [np.cos(dt/2), -1j*np.sin(dt/2)],
    [-1j*np.sin(dt/2), np.cos(dt/2)],
], dtype=complex))
velocity_h = (rho_rotated_small - rho0) / dt

tangent_h, normal_h = tangent_normal_decomposition(rho0, velocity_h)
print("Unitary tangent norm:", np.linalg.norm(tangent_h))
print("Unitary spectral-block norm:", np.linalg.norm(normal_h))

rho_damped_small = amplitude_damping_channel(rho0, dt)
velocity_d = (rho_damped_small - rho0) / dt

tangent_d, normal_d = tangent_normal_decomposition(rho0, velocity_d)
print("Damping tangent norm:", np.linalg.norm(tangent_d))
print("Damping spectral-block norm:", np.linalg.norm(normal_d))

## 4. Lyapunov generator

The horizontal generator in the reviewed framework solves

$$G\rho + \rho G = \dot\rho.$$

We verify the residual numerically.

In [ ]:
g = lyapunov_generator(rho0, velocity_d)
residual = g @ rho0 + rho0 @ g - velocity_d
print("Lyapunov residual Frobenius norm:", np.linalg.norm(residual))
print("Generator Hermiticity error:", np.linalg.norm(g - g.conj().T))

## 5. Entropy and path information

Entropy is an endpoint/state statistic. A path-transport factor is order-sensitive. It should not be interpreted as a lossless record of the complete history.

In [ ]:
print("Initial entropy:", von_neumann_entropy(rho0), "bits")
print("Entropy R -> D:", von_neumann_entropy(rho_rotation_then_damping), "bits")
print("Entropy D -> R:", von_neumann_entropy(rho_damping_then_rotation), "bits")

## 6. Discrete Uhlmann transport on a closed loop

The path below is a smooth loop inside the Bloch ball. Consecutive purification amplitudes are aligned using the polar unitary of $\sqrt{\rho_{k+1}}\sqrt{\rho_k}$.

In [ ]:
t = np.linspace(0, 2*np.pi, 160)
loop = [bloch_to_density([0.28*np.cos(s), 0.18*np.sin(s), 0.20 + 0.08*np.sin(2*s)]) for s in t]
loop[-1] = loop[0]
transport, steps_u = discrete_uhlmann_transport(loop)
print("Closed-loop transport unitary:")
print(transport)
print("Unitarity residual:", np.linalg.norm(transport.conj().T @ transport - np.eye(2)))
print("Deviation from identity:", np.linalg.norm(transport - np.eye(2)))

## Engineering takeaway

- Endpoint metrics can miss curriculum or channel-order sensitivity.
- Tangent-like change suggests reorientation inside a regime.
- Spectral-block change suggests structural drift, specialization, forgetting, or regime transition.
- Path-aware metrics complement rather than replace accuracy, entropy, MMD, Fisher-Rao/Bures distance, and operational validation.